# 📊 Ablation Significance Analysis — ArchPipeline vs Direct Evolution

This notebook addresses the reviewer's comment on the robustness of the ablation study:
- Per-instance scores (not just the mean) for both models
- Statistical significance testing (paired Wilcoxon signed-rank test)
- Bootstrap confidence interval on the score difference
- Qualitative examples at the operation level
- "Architectural vs. lexical" analysis via operational precision and syntactic validity

**Prerequisites:** the trained checkpoints already present on Google Drive (`OUT_CT5B_A` for ArchPipeline, `evo_ablation_DirectEvolution_CodeT5base_FullFT` for the Ablation Study — both produced by `Evolution_Finetuning.ipynb`), and the `evo_test_v2.jsonl` file (120 instances).

In [ ]:
# ============================================================
# STEP 0 — Install dependencies
# ============================================================
!pip install -q transformers torch nltk rouge-score bert-score scipy

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✅ Dependencies installed")

In [ ]:
# ============================================================
# STEP 1 — Google Drive mount & paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# --- Paths (identical to Evolution_Finetuning.ipynb) ---
OUT_CT5B_A      = "/content/drive/MyDrive/evo_CodeT5base_fullFT"
OUT_ABLATION    = "/content/drive/MyDrive/evo_ablation_DirectEvolution_CodeT5base_FullFT"
EVO_TEST_PATH   = "/content/drive/MyDrive/dataset_clean/evolution/evo_test_v2.jsonl"

# --- Path where this notebook's results are saved ---
SAVE_DIR = f"{OUT_CT5B_A}/ablation_significance_analysis"
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Results will be saved to: {SAVE_DIR}")

In [ ]:
# ============================================================
# STEP 2 — Load the test set (120 instances)
# ============================================================
import json

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

test_data = load_jsonl(EVO_TEST_PATH)
for i, ex in enumerate(test_data):
    ex["_id"] = i

print(f"📊 Test set loaded: {len(test_data)} instances")
from collections import Counter
ops_count = Counter(ex["operation"] for ex in test_data)
for op, n in sorted(ops_count.items()):
    print(f"   {op:<25}: {n}")

In [ ]:
# ============================================================
# STEP 3 — Load the two models being compared
# ============================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("⏳ Loading ArchPipeline (CodeT5-base StratA, with transfer)...")
tok_archpipeline = AutoTokenizer.from_pretrained(OUT_CT5B_A)
model_archpipeline = AutoModelForSeq2SeqLM.from_pretrained(
    OUT_CT5B_A, torch_dtype=torch.float32).to(device)
model_archpipeline.eval()
print("✅ ArchPipeline loaded")

print("\n⏳ Loading Ablation Study (Direct Evolution, without transfer)...")
tok_ablation = AutoTokenizer.from_pretrained(OUT_ABLATION)
model_ablation = AutoModelForSeq2SeqLM.from_pretrained(
    OUT_ABLATION, torch_dtype=torch.float32).to(device)
model_ablation.eval()
print("✅ Ablation Study loaded")

In [ ]:
# ============================================================
# STEP 4 — Generation + PER-INSTANCE scores (not just the mean)
# ============================================================
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn

scorer_r = rs_module.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method4

def generate_predictions(model, tokenizer, test_data, model_name):
    """Generates the model's predictions on the test set, in order."""
    model.eval()
    predictions = []
    print(f"\n🔄 Generating — {model_name} ({len(test_data)} examples)...")
    for i, ex in enumerate(test_data):
        inputs = tokenizer(
            str(ex["input"]), return_tensors="pt",
            max_length=256, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_length=256, num_beams=4,
                early_stopping=True, no_repeat_ngram_size=3,
                length_penalty=1.0)
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        predictions.append(pred)
        if (i + 1) % 30 == 0:
            print(f"   {i+1}/{len(test_data)} generated...")
    print(f"✅ {len(predictions)} predictions — {model_name}")
    return predictions


def compute_per_instance_scores(predictions, references):
    """Computes BLEU, ROUGE-L, METEOR, EM for EACH instance (not the mean)."""
    em_per, bleu_per, rougeL_per, meteor_per = [], [], [], []
    for p, r in zip(predictions, references):
        p, r = p.strip(), r.strip()
        em_per.append(1.0 if p == r else 0.0)
        bleu_per.append(sentence_bleu([r.split()], p.split(), smoothing_function=smooth))
        rougeL_per.append(scorer_r.score(r, p)["rougeL"].fmeasure)
        try:
            meteor_per.append(meteor_score([r.split()], p.split()))
        except Exception:
            meteor_per.append(0.0)

    print("   🔄 Computing BERTScore (per instance)...")
    _, _, bert_f1 = bert_score_fn(predictions, references, lang="en",
                                   rescale_with_baseline=False, verbose=False)
    bert_f1_per = bert_f1.numpy().tolist()

    return {
        "EM": em_per, "BLEU": bleu_per,
        "ROUGE-L": rougeL_per, "METEOR": meteor_per,
        "BERT-F1": bert_f1_per
    }

print("✅ Functions defined")

In [ ]:
# ============================================================
# STEP 5 — Run generation + per-instance scoring for both models
# ============================================================
references = [str(ex["target"]).strip() for ex in test_data]

preds_archpipeline = generate_predictions(
    model_archpipeline, tok_archpipeline, test_data, "ArchPipeline (StratA)")
scores_archpipeline = compute_per_instance_scores(preds_archpipeline, references)

preds_ablation = generate_predictions(
    model_ablation, tok_ablation, test_data, "Ablation (Direct Evolution)")
scores_ablation = compute_per_instance_scores(preds_ablation, references)

print("\n✅ Per-instance scores computed for both models")

In [ ]:
# ============================================================
# STEP 5B — Save per-instance results to disk
#
# ⚠️ RECONSTRUCTED CELL. Two later cells in this notebook (the manual
# review extraction and the ADD_COMPONENT deep dive) load a
# `per_instance_scores.json` file that is never written anywhere in
# the original notebook — the cell that produced it was apparently
# lost. It is reconstructed here, right after generation, so the
# expensive GPU predictions above are checkpointed to disk and the
# later cells have the file they expect. The schema (one entry per
# test instance, with `archpipeline`/`ablation` sub-dicts each
# containing `prediction` + per-metric scores) was inferred from how
# those later cells read the file.
# ============================================================
per_instance_data = []
for i, ex in enumerate(test_data):
    entry = {
        "id"        : ex["_id"],
        "operation" : ex["operation"],
        "adl_type"  : ex["adl_type"],
        "reference" : references[i],
        "archpipeline": {
            "prediction": preds_archpipeline[i],
            **{metric: scores_archpipeline[metric][i] for metric in scores_archpipeline},
        },
        "ablation": {
            "prediction": preds_ablation[i],
            **{metric: scores_ablation[metric][i] for metric in scores_ablation},
        },
    }
    per_instance_data.append(entry)

with open(f"{SAVE_DIR}/per_instance_scores.json", "w", encoding="utf-8") as f:
    json.dump(per_instance_data, f, indent=2, ensure_ascii=False)

print(f"💾 Saved: {SAVE_DIR}/per_instance_scores.json ({len(per_instance_data)} instances)")

In [ ]:
# ============================================================
# STEP 6 — Statistical significance testing (paired Wilcoxon)
# + bootstrap confidence interval on the difference
# ============================================================
from scipy.stats import wilcoxon

def bootstrap_ci_diff(scores_a, scores_b, B=2000, seed=42, ci=95):
    """Bootstrap CI on the mean difference (a - b), paired per instance."""
    rng = np.random.default_rng(seed)
    a = np.array(scores_a)
    b = np.array(scores_b)
    n = len(a)
    diffs = a - b
    boot_means = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        boot_means.append(diffs[idx].mean())
    boot_means = np.array(boot_means)
    lo = np.percentile(boot_means, (100 - ci) / 2)
    hi = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return diffs.mean(), lo, hi


print("="*80)
print("📊 SIGNIFICANCE TESTING — ArchPipeline vs Direct Evolution (Ablation)")
print("   Paired comparison on n=120 test instances")
print("="*80)
print(f"{'Metric':<12}{'Mean Δ':>10}{'95% CI':>22}{'Wilcoxon p':>14}{'Signif.':>10}")
print("-"*80)

significance_results = {}
for metric in ["EM", "BLEU", "ROUGE-L", "METEOR", "BERT-F1"]:
    a = scores_archpipeline[metric]
    b = scores_ablation[metric]

    mean_diff, ci_lo, ci_hi = bootstrap_ci_diff(a, b)

    # Wilcoxon: pairs with diff=0 are dropped automatically (zero_method='wilcox')
    try:
        stat, p_value = wilcoxon(a, b, zero_method="wilcox")
    except ValueError:
        # All differences are zero
        p_value = 1.0

    signif = "✅ p<0.05" if p_value < 0.05 else "❌ n.s."
    print(f"{metric:<12}{mean_diff:>10.4f}  [{ci_lo:>7.4f}, {ci_hi:>7.4f}]{p_value:>14.4f}{signif:>10}")

    significance_results[metric] = {
        "mean_diff": float(mean_diff),
        "ci_95_lower": float(ci_lo),
        "ci_95_upper": float(ci_hi),
        "wilcoxon_p_value": float(p_value),
        "significant_at_0.05": bool(p_value < 0.05)
    }

print("="*80)

# ── Save for the paper / supplementary material ────────────────
with open(f"{SAVE_DIR}/significance_results.json", "w", encoding="utf-8") as f:
    json.dump(significance_results, f, indent=2, ensure_ascii=False)
print(f"\n💾 Saved: {SAVE_DIR}/significance_results.json")

In [ ]:
# ============================================================
# STEP 7 — Qualitative examples: cases where ArchPipeline and
# Ablation differ. Focus on ADD_COMPONENT (the weakest operation
# for both configurations)
# ============================================================
print("🔍 Cases where the two models produce DIFFERENT outputs\n")

qualitative_cases = []
for i, ex in enumerate(test_data):
    pred_a = preds_archpipeline[i].strip()
    pred_b = preds_ablation[i].strip()
    ref    = references[i].strip()

    if pred_a != pred_b:
        case = {
            "id": ex["_id"],
            "operation": ex["operation"],
            "adl_type": ex["adl_type"],
            "input": ex["input"],
            "reference": ref,
            "archpipeline_pred": pred_a,
            "ablation_pred": pred_b,
            "archpipeline_correct": pred_a == ref,
            "ablation_correct": pred_b == ref,
        }
        qualitative_cases.append(case)

print(f"Total instances with differing predictions: {len(qualitative_cases)} / {len(test_data)}\n")

# Show 3 representative ADD_COMPONENT examples (the weakest operation)
add_component_cases = [c for c in qualitative_cases if c["operation"] == "ADD_COMPONENT"]
print(f"ADD_COMPONENT cases with a difference: {len(add_component_cases)}\n")

for case in add_component_cases[:3]:
    print("─"*70)
    print(f"ID={case['id']} | {case['operation']} | {case['adl_type']}")
    print(f"Reference         : {case['reference'][:100]}")
    print(f"ArchPipeline      : {case['archpipeline_pred'][:100]}  {'✅' if case['archpipeline_correct'] else '❌'}")
    print(f"Ablation (Direct) : {case['ablation_pred'][:100]}  {'✅' if case['ablation_correct'] else '❌'}")
print("─"*70)

# ── Save the full list of differing cases (paper supplementary material) ──
with open(f"{SAVE_DIR}/qualitative_differing_cases.json", "w", encoding="utf-8") as f:
    json.dump(qualitative_cases, f, indent=2, ensure_ascii=False)
print(f"\n💾 Saved: {SAVE_DIR}/qualitative_differing_cases.json")

In [ ]:
# ============================================================
# STEP 7B — Cases where ONLY ONE of the two models succeeds
# (the most telling cases for showing the effect of the transfer)
# ============================================================

# Cases where ArchPipeline succeeds BUT Ablation fails
archpipeline_wins = [c for c in qualitative_cases
                     if c["archpipeline_correct"] and not c["ablation_correct"]]

# Cases where Ablation succeeds BUT ArchPipeline fails (the reverse — also worth examining)
ablation_wins = [c for c in qualitative_cases
                  if c["ablation_correct"] and not c["archpipeline_correct"]]

print(f"✅ ArchPipeline succeeds, Ablation fails: {len(archpipeline_wins)} cases")
print(f"⚠️  Ablation succeeds, ArchPipeline fails: {len(ablation_wins)} cases")
print()

print("="*70)
print("CASES WHERE THE TRANSFER MAKES THE DIFFERENCE (ArchPipeline ✅ / Ablation ❌)")
print("="*70)
for case in archpipeline_wins[:5]:
    print("─"*70)
    print(f"ID={case['id']} | {case['operation']} | {case['adl_type']}")
    print(f"Reference         : {case['reference'][:100]}")
    print(f"ArchPipeline      : {case['archpipeline_pred'][:100]}  ✅")
    print(f"Ablation (Direct) : {case['ablation_pred'][:100]}  ❌")
print("─"*70)

if ablation_wins:
    print("\n" + "="*70)
    print("REVERSE CASES (Ablation ✅ / ArchPipeline ❌) — included for transparency")
    print("="*70)
    for case in ablation_wins[:5]:
        print("─"*70)
        print(f"ID={case['id']} | {case['operation']} | {case['adl_type']}")
        print(f"Reference         : {case['reference'][:100]}")
        print(f"ArchPipeline      : {case['archpipeline_pred'][:100]}  ❌")
        print(f"Ablation (Direct) : {case['ablation_pred'][:100]}  ✅")
    print("─"*70)

# Breakdown by operation of the cases where ArchPipeline wins
from collections import Counter
ops_wins = Counter(c["operation"] for c in archpipeline_wins)
print(f"\n📊 Breakdown by operation (ArchPipeline wins):")
for op, n in sorted(ops_wins.items(), key=lambda x: -x[1]):
    print(f"   {op:<25}: {n}")

## Step 8 — Architectural vs. Lexical: Syntactic Validity & Operational Precision

To show that the gain is not "purely lexical", this step reuses the `compute_syntactic_validity()` and `compute_operational_precision()` functions from `Evolution_Finetuning.ipynb`'s evaluation pipeline (Step 9 of that notebook).

They are duplicated inline in the next cell (rather than imported) so this notebook stays self-contained and runnable without needing `Evolution_Finetuning.ipynb` to be executed first. The logic is copied verbatim from that notebook's cleaned version to avoid any divergence from the code used to produce the paper's results — if you update one copy, update the other.

In [ ]:
# ============================================================
# STEP 8 — Syntactic validity & operational precision (per model)
# Duplicated from Evolution_Finetuning.ipynb's evaluation pipeline
# (Step 7-8) — see the markdown note above.
# ============================================================
import re

def clean_field(value) -> str:
    if value is None:
        return ""
    s = str(value).strip()
    return "" if s.lower() in ("none", "nan", "null", "") else s


def verify_operation(pred: str, example: dict) -> bool:
    op       = example.get("operation", "").upper()
    target   = clean_field(example.get("cible", ""))
    pred_l   = pred.lower().strip()
    input_l  = str(example.get("input", "")).lower()

    if op == "ADD_PORT":
        ports_before = len(re.findall(r'port\s+\w+', input_l))
        ports_after  = len(re.findall(r'port\s+\w+', pred_l))
        if ports_after > ports_before:
            return True
        return bool(re.search(r'port\s+\w+', pred_l))

    elif op == "MODIFY_PROPERTY":
        props_before = set(re.findall(r'[\w:]+\s*[=>]+\s*[\w".]+', input_l))
        props_pred   = set(re.findall(r'[\w:]+\s*[=>]+\s*[\w".]+', pred_l))
        return len(props_pred - props_before) > 0

    elif op == "ADD_COMPONENT":
        comps_before = len(re.findall(r'component\s+\w+', input_l, re.IGNORECASE))
        comps_after  = len(re.findall(r'component\s+\w+', pred_l, re.IGNORECASE))
        if comps_after > comps_before:
            return True
        return bool(re.search(r'subcomponents\s+\w+', pred_l, re.IGNORECASE))

    elif op == "DELETE_COMPONENT":
        if not target:
            return len(pred_l.strip()) > 5
        return target.lower() not in pred_l

    elif op == "ADD_CONNECTOR":
        # ── FIXED VERSION V5 ──────────────────────────────────
        # Only checks that the prediction contains a connector AND
        # differs from the source, without requiring an exact match
        # with the reference target.
        CONNECTOR_KEYWORDS = [
            "connector", "connections", "bus access",
            "pipe",      "channel",     "bus",
            "link",      "bridge",      "eventchannel",
            "asyncpipe", "msgqueue",    "databus",
            "eventbus",  "ctrllink",    "servicebus",
            "netlink",   "sharedbus",   "streampipe",
            "httpconn",  "databridg",   "servicelink",
            "rpcconn",   "asynclink",   "syncbus",
        ]
        # Extract the source ADL from the input
        if "[adl]" in input_l:
            adl_source = input_l.split("[adl]")[-1]\
                                 .split("[req]")[0].strip()
        else:
            adl_source = input_l

        pred_has_conn    = any(kw in pred_l for kw in CONNECTOR_KEYWORDS)
        pred_diff_source = pred_l.strip() != adl_source.strip()
        return pred_has_conn and pred_diff_source

    elif op == "DELETE_CONNECTOR":
        if not target:
            return len(pred_l.strip()) > 5
        return target.lower() not in pred_l

    return False


KEYWORDS_ADL = {
    "component", "connector", "system", "style", "family",
    "port", "role", "property", "attachment", "binding",
    "subcomponent", "thread", "process", "porttype",
    "datatype", "eventlist", "implementation",
}

def is_syntactically_valid(adl: str) -> bool:
    if not adl or len(adl.strip()) < 5:
        return False
    if adl.count("{") != adl.count("}"):
        return False
    if re.search(r'\.{3}|…', adl):
        return False
    return any(kw in adl.lower() for kw in KEYWORDS_ADL)


def compute_syntactic_validity(predictions: list, test_data: list) -> dict:
    valid_flags = [is_syntactically_valid(p) for p in predictions]
    acme, aadl = [], []
    for i, ex in enumerate(test_data):
        t = ex.get("adl_type", "").upper()
        if   t == "ACME": acme.append(valid_flags[i])
        elif t == "AADL": aadl.append(valid_flags[i])
    return {
        "global_validity": float(np.mean(valid_flags)),
        "validity_ACME"  : float(np.mean(acme)) if acme else 0.0,
        "validity_AADL"  : float(np.mean(aadl)) if aadl else 0.0,
        "n_valid"        : int(sum(valid_flags)),
        "n_total"        : len(predictions),
    }


def compute_operational_precision(predictions: list, test_data: list) -> dict:
    results = {}
    total_correct, total_count = 0, 0

    for pred, ex in zip(predictions, test_data):
        op      = ex.get("operation", "UNKNOWN")
        correct = verify_operation(pred, ex)
        if op not in results:
            results[op] = {"correct": 0, "total": 0}
        results[op]["correct"] += int(correct)
        results[op]["total"]   += 1
        total_correct += int(correct)
        total_count   += 1

    for op in results:
        c = results[op]["correct"]
        t = results[op]["total"]
        results[op]["precision"] = round(c / t, 4) if t > 0 else 0.0

    results["GLOBAL"] = {
        "correct"  : total_correct,
        "total"    : total_count,
        "precision": round(total_correct / total_count, 4)
                     if total_count > 0 else 0.0,
    }
    return results


validity_archpipeline  = compute_syntactic_validity(preds_archpipeline, test_data)
precision_archpipeline = compute_operational_precision(preds_archpipeline, test_data)

validity_ablation  = compute_syntactic_validity(preds_ablation, test_data)
precision_ablation = compute_operational_precision(preds_ablation, test_data)

print("ArchPipeline — Global validity:", validity_archpipeline.get("global_validity"))
print("Ablation     — Global validity:", validity_ablation.get("global_validity"))
print()
print("ArchPipeline — Operational precision:", precision_archpipeline.get("GLOBAL"))
print("Ablation     — Operational precision:", precision_ablation.get("GLOBAL"))

In [ ]:
# ============================================================
# REVIEWER RESPONSE — Stronger validation: checking that the exact
# requested identifier is preserved (not just the element type)
# ============================================================
import re

def extract_target_identifier(req: str, op: str):
    """Extracts the exact expected identifier from the [REQ] field of
    the input, using patterns specific to each operation type.
    Manually validated: 120/120 (100%) of the test set instances."""
    req = req.strip()
    patterns = {
        "ADD_PORT": [
            r"(?:Insert a new|Augment \w+ with a new|Attach a|Extend \w+ with a)\s+(\w+)\s+(?:port|interface)",
            r"(?:a|an)\s+(\w+)\s+(?:port|interface)",
        ],
        "ADD_COMPONENT": [
            r"named\s+(\w+)",
            r"(?:Insert a new|Add a new)\s+(\w+)\s+(?:component|of type)",
            r"Introduce\s+a\s+(\w+)\s+\w+\s+component",
            r"(?:a|an)\s+\w+\s+component\s+(?:named\s+)?(\w+)",
        ],
        "ADD_CONNECTOR": [
            r"(?:a\s+new|a|an)\s+(\w+)\s+connector",
        ],
        "MODIFY_PROPERTY": [
            r"[Uu]pdate\s+(\w+)",
            r"[Ss]et\s+\w+\s+of\s+(\w+)",
            r"[Cc]hange\s+(\w+)'s",
            r"[Ss]et\s+the\s+\w+\s+property\s+of\s+(\w+)",
            r"[Cc]hange\s+the\s+\w+\s+of\s+(\w+)",
            r"[Mm]odify\s+the\s+\w+\s+attribute\s+of\s+(\w+)",
        ],
        "DELETE_COMPONENT": [
            r"component\s+(\w+)",
            r"[Dd]elete\s+(\w+)\s+from",
            r"[Rr]emove\s+(\w+)\s+from",
        ],
        "DELETE_CONNECTOR": [
            r"connector\s+(\w+)",
            r"[Dd]elete\s+the\s+(\w+)\s+connector",
        ],
    }
    for pat in patterns.get(op, []):
        m = re.search(pat, req)
        if m:
            return m.group(1)
    return None


def verify_operation_v2(pred: str, example: dict) -> dict:
    """Stronger version of verify_operation().
    Returns a detailed dict distinguishing:
    - 'type_correct'       : was the expected element type added/modified/deleted?
    - 'identifier_correct' : is the exact requested IDENTIFIER present
                              (for add) / absent (for delete) in the prediction?
    - 'correct'            : both conditions combined (strict criterion)
    """
    op      = example.get("operation", "").upper()
    pred_l  = pred.lower().strip()
    input_l = str(example.get("input", "")).lower()
    req     = str(example.get("input", "")).split("[REQ]")[-1].strip()

    identifier = extract_target_identifier(req, op)
    id_l = identifier.lower() if identifier else None

    # --- Check the element TYPE (original logic, kept as-is) ---
    type_correct = verify_operation(pred, example)  # original function

    # --- Check the EXACT identifier ---
    if id_l is None:
        identifier_correct = None  # not checkable, no penalty
    elif op in ("ADD_PORT", "ADD_COMPONENT", "ADD_CONNECTOR", "MODIFY_PROPERTY"):
        # The identifier must be PRESENT in the prediction
        identifier_correct = id_l in pred_l
    elif op in ("DELETE_COMPONENT", "DELETE_CONNECTOR"):
        # The identifier must be ABSENT from the prediction (successful deletion)
        identifier_correct = id_l not in pred_l
    else:
        identifier_correct = None

    correct_strict = type_correct and (identifier_correct in (True, None))

    return {
        "type_correct": type_correct,
        "identifier_correct": identifier_correct,
        "expected_identifier": identifier,
        "correct": correct_strict,
    }


def compute_operational_precision_v2(predictions: list, test_data: list) -> dict:
    """Stronger version: operational precision decomposed into
    type-level vs. identifier-level, per operation."""
    results = {}
    global_stats = {"type_correct": 0, "identifier_correct": 0,
                     "both_correct": 0, "total": 0,
                     "identifier_checkable": 0}

    for pred, ex in zip(predictions, test_data):
        op = ex.get("operation", "UNKNOWN")
        r = verify_operation_v2(pred, ex)

        if op not in results:
            results[op] = {"type_correct": 0, "identifier_correct": 0,
                            "both_correct": 0, "total": 0,
                            "identifier_checkable": 0}

        results[op]["total"] += 1
        global_stats["total"] += 1

        results[op]["type_correct"] += int(r["type_correct"])
        global_stats["type_correct"] += int(r["type_correct"])

        if r["identifier_correct"] is not None:
            results[op]["identifier_checkable"] += 1
            global_stats["identifier_checkable"] += 1
            results[op]["identifier_correct"] += int(r["identifier_correct"])
            global_stats["identifier_correct"] += int(r["identifier_correct"])

        results[op]["both_correct"] += int(r["correct"])
        global_stats["both_correct"] += int(r["correct"])

    for op in results:
        t = results[op]["total"]
        ic = results[op]["identifier_checkable"]
        results[op]["precision_type"] = round(results[op]["type_correct"] / t, 4) if t else 0.0
        results[op]["precision_identifier"] = (
            round(results[op]["identifier_correct"] / ic, 4) if ic else None)
        results[op]["precision_strict"] = round(results[op]["both_correct"] / t, 4) if t else 0.0

    t, ic = global_stats["total"], global_stats["identifier_checkable"]
    global_stats["precision_type"] = round(global_stats["type_correct"] / t, 4) if t else 0.0
    global_stats["precision_identifier"] = (
        round(global_stats["identifier_correct"] / ic, 4) if ic else None)
    global_stats["precision_strict"] = round(global_stats["both_correct"] / t, 4) if t else 0.0

    results["GLOBAL"] = global_stats
    return results


print("✅ Strengthened validation functions (exact identifier) loaded")
print("   - extract_target_identifier()")
print("   - verify_operation_v2()")
print("   - compute_operational_precision_v2()")

In [ ]:
# Original version (type only, permissive)
precision_archpipeline = compute_operational_precision(preds_archpipeline, test_data)
precision_ablation = compute_operational_precision(preds_ablation, test_data)

# New strict version (type + exact identifier)
precision_archpipeline_v2 = compute_operational_precision_v2(preds_archpipeline, test_data)
precision_ablation_v2 = compute_operational_precision_v2(preds_ablation, test_data)

print("=== Original precision (type only) ===")
print("ArchPipeline GLOBAL:", precision_archpipeline["GLOBAL"])
print("Ablation     GLOBAL:", precision_ablation["GLOBAL"])
print()
print("=== Strict precision (type + exact identifier) ===")
print("ArchPipeline GLOBAL:", precision_archpipeline_v2["GLOBAL"])
print("Ablation     GLOBAL:", precision_ablation_v2["GLOBAL"])
print()
print("=== Per-operation detail (strict) ===")
for op, vals in precision_archpipeline_v2.items():
    if op != "GLOBAL":
        print(f"ArchPipeline {op:<20}: type={vals['precision_type']}, "
              f"id={vals['precision_identifier']}, strict={vals['precision_strict']}")
for op, vals in precision_ablation_v2.items():
    if op != "GLOBAL":
        print(f"Ablation     {op:<20}: type={vals['precision_type']}, "
              f"id={vals['precision_identifier']}, strict={vals['precision_strict']}")

In [ ]:
# ============================================================
# Save the operational precision results
# (original + strict with exact identifier)
# ============================================================
from datetime import datetime

operational_precision_results = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "n": 120,
        "description": "Comparison of type-level (original) vs. identifier-level (strict) "
                        "operational precision, in response to the reviewer's request for "
                        "stronger architectural-correctness validation."
    },
    "type_level_original": {
        "ArchPipeline": precision_archpipeline,
        "Direct_Evolution": precision_ablation,
    },
    "strict_identifier_level": {
        "ArchPipeline": precision_archpipeline_v2,
        "Direct_Evolution": precision_ablation_v2,
    }
}

with open(f"{SAVE_DIR}/operational_precision_strict_vs_original.json", "w", encoding="utf-8") as f:
    json.dump(operational_precision_results, f, indent=2, ensure_ascii=False)

# Human-readable text version (README-style)
with open(f"{SAVE_DIR}/operational_precision_strict_vs_original.txt", "w", encoding="utf-8") as f:
    f.write("Operational Precision — Type-level (original) vs. Identifier-level (strict)\n")
    f.write("="*80 + "\n\n")
    f.write("GLOBAL RESULTS\n")
    f.write("-"*80 + "\n")
    f.write(f"{'Config':<20}{'Type-level':>15}{'Identifier-level':>20}{'Strict (both)':>18}\n")
    f.write(f"{'ArchPipeline':<20}{precision_archpipeline['GLOBAL']['precision']:>15.4f}"
            f"{precision_archpipeline_v2['GLOBAL']['precision_identifier']:>20.4f}"
            f"{precision_archpipeline_v2['GLOBAL']['precision_strict']:>18.4f}\n")
    f.write(f"{'Direct Evolution':<20}{precision_ablation['GLOBAL']['precision']:>15.4f}"
            f"{precision_ablation_v2['GLOBAL']['precision_identifier']:>20.4f}"
            f"{precision_ablation_v2['GLOBAL']['precision_strict']:>18.4f}\n\n")

    f.write("PER-OPERATION RESULTS (strict criterion)\n")
    f.write("-"*80 + "\n")
    f.write(f"{'Operation':<20}{'ArchPipeline':>15}{'Direct Evolution':>20}\n")
    for op in precision_archpipeline_v2:
        if op != "GLOBAL":
            a = precision_archpipeline_v2[op]['precision_strict']
            b = precision_ablation_v2[op]['precision_strict']
            f.write(f"{op:<20}{a:>15.4f}{b:>20.4f}\n")

print(f"💾 Results saved to: {SAVE_DIR}")
print("   - operational_precision_strict_vs_original.json")
print("   - operational_precision_strict_vs_original.txt")

In [ ]:
# ============================================================
# STEP B — Extract 20 instances for manual review
# (architectural error categories)
# ============================================================
with open(f"{SAVE_DIR}/per_instance_scores.json", "r", encoding="utf-8") as f:
    per_instance_data = json.load(f)

TARGET_IDS = [0, 5, 6, 8, 15, 17, 18, 21, 22, 25, 26, 36, 41, 44, 46, 86, 88, 102, 114, 119]

# Reload the original file to get the full "input" field (including [REQ])
raw_test_data = load_jsonl(EVO_TEST_PATH)

# Build the manual-review table
manual_review_data = []
for item in per_instance_data:
    if item["id"] in TARGET_IDS:
        raw_ex = raw_test_data[item["id"]]
        req = raw_ex["input"].split("[REQ]")[-1].strip()
        manual_review_data.append({
            "id": item["id"],
            "operation": item["operation"],
            "adl_type": item["adl_type"],
            "request": req,
            "reference": item["reference"],
            "archpipeline_pred": item["archpipeline"]["prediction"],
            "ablation_pred": item["ablation"]["prediction"],
        })

manual_review_data.sort(key=lambda x: x["id"])

with open(f"{SAVE_DIR}/manual_review_20instances.json", "w", encoding="utf-8") as f:
    json.dump(manual_review_data, f, indent=2, ensure_ascii=False)

print(f"✅ {len(manual_review_data)} instances extracted for manual review")
print(f"💾 Saved: {SAVE_DIR}/manual_review_20instances.json")
print()

# Formatted display to make the review easier
for item in manual_review_data:
    print("="*90)
    print(f"ID={item['id']} | {item['operation']} | {item['adl_type']}")
    print(f"REQUEST     : {item['request']}")
    print(f"REFERENCE   : {item['reference']}")
    print(f"ARCHPIPELINE: {item['archpipeline_pred']}")
    print(f"ABLATION    : {item['ablation_pred']}")
print("="*90)

In [ ]:
# ============================================================
# In-depth ADD_COMPONENT analysis — all 20 instances (full n=120)
# Response to the reviewer's comment on the ADD_COMPONENT failure mode
# ============================================================
with open(f"{SAVE_DIR}/per_instance_scores.json", "r", encoding="utf-8") as f:
    per_instance_data = json.load(f)

raw_test_data = load_jsonl(EVO_TEST_PATH)

# Filter ADD_COMPONENT
add_component_cases = []
for item in per_instance_data:
    if item["operation"] == "ADD_COMPONENT":
        raw_ex = raw_test_data[item["id"]]
        req = raw_ex["input"].split("[REQ]")[-1].strip()
        add_component_cases.append({
            "id": item["id"],
            "adl_type": item["adl_type"],
            "request": req,
            "reference": item["reference"],
            "archpipeline_pred": item["archpipeline"]["prediction"],
            "ablation_pred": item["ablation"]["prediction"],
            "archpipeline_em": item["archpipeline"]["EM"],
            "ablation_em": item["ablation"]["EM"],
        })

add_component_cases.sort(key=lambda x: x["id"])
print(f"📊 Total ADD_COMPONENT instances: {len(add_component_cases)}")
print(f"   ArchPipeline EM=1.0: {sum(c['archpipeline_em']==1.0 for c in add_component_cases)}/{len(add_component_cases)}")
print(f"   Ablation     EM=1.0: {sum(c['ablation_em']==1.0 for c in add_component_cases)}/{len(add_component_cases)}")
print()

# Save for the full analysis
with open(f"{SAVE_DIR}/add_component_all_20_cases.json", "w", encoding="utf-8") as f:
    json.dump(add_component_cases, f, indent=2, ensure_ascii=False)

# Show every case where ArchPipeline fails (EM != 1.0)
print("="*90)
print("ArchPipeline FAILURE CASES (ADD_COMPONENT, EM != 1.0)")
print("="*90)
for c in add_component_cases:
    if c["archpipeline_em"] != 1.0:
        print(f"ID={c['id']} | {c['adl_type']}")
        print(f"REQUEST  : {c['request']}")
        print(f"REFERENCE: {c['reference'][:150]}")
        print(f"ARCHPIPE.: {c['archpipeline_pred'][:150]}")
        print("-"*90)

print(f"\n💾 Saved: {SAVE_DIR}/add_component_all_20_cases.json")